# 构建一个单行代码的研究智能体

研究任务会消耗专家的大量时间：市场分析师手工收集竞争情报，法务团队跟踪法规变化，工程师在各类文档中排查缺陷报告。核心难点不在于找到信息，而在于根据刚刚发现的内容判断下一步该搜索什么。

Claude Agent SDK 让构建能够自主探索外部系统的智能体成为可能，而且无需预先定义的工作流。传统工作流自动化按固定步骤执行，而研究智能体会根据所发现的内容调整策略——追踪有希望的线索、综合相互冲突的信息来源，并判断何时已获得足够信息来回答问题。

## 完成本 cookbook 后，你将能够：

- 用几行代码构建一个能自主搜索并综合信息的智能体

这一基础适用于所有无法预先获得所需信息的任务：竞争分析、技术故障排查、投资研究或文献综述。

# 为什么选择研究智能体？

研究是理想的智能体用例，原因有二：

1. **信息不是自包含的**。仅凭输入的问题并不包含答案。智能体必须与外部系统（搜索引擎、数据库、API）交互来收集所需内容。
2. **路径在探索过程中浮现**。你无法预先确定工作流。智能体应该搜索公司财务数据还是监管备案文件，取决于它对商业模式发现了什么。最优策略会在调查过程中自行显现。

最简单的形式下，研究智能体搜索网络并综合所获结果。下面我们用 Claude Agent SDK 内置的网络搜索工具，仅用几行代码构建正是这样的智能体。

说明：你也可以查看 [Claude Code 内置工具的完整列表](https://docs.claude.com/en/docs/claude-code/settings#tools-available-to-claude)

# 前置条件

在跟随本指南操作之前，请确认你已具备：

**必备知识**

* Python 基础——熟悉 async/await、函数和基本数据结构
* 对智能体模式有基本理解——如果你是智能体领域的新手，建议先阅读[构建高效智能体](https://www.anthropic.com/engineering/building-effective-agents)

**必备工具**

* Python 3.11 或更高版本
* Anthropic API 密钥[(在此获取)](https://console.anthropic.com)

**建议具备：**
* 熟悉 Claude Agent SDK 的概念
* 理解 LLM 中的工具使用模式


## 环境准备

首先，安装所需的依赖：

In [ ]:
%%capture
%pip install -U claude-agent-sdk python-dotenv

说明：请确保你的 .env 文件包含：

```bash
ANTHROPIC_API_KEY=your_key_here
```

加载环境变量并配置客户端：

In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-opus-4-6"

## 构建你的第一个研究智能体

我们从最简单的实现开始：一个能搜索网络并综合所获结果的研究智能体。使用 Claude Agent SDK，这只需要几行代码。

关键在于 query（）函数，它创建一次无状态的智能体交互。我们只给 Claude 提供一个工具 WebSearch，让它根据我们的研究问题自主决定何时以及如何使用它。

In [3]:
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
)

from claude_agent_sdk import ClaudeAgentOptions, query

messages = []
async for msg in query(
    prompt="Research the latest trends in AI agents and give me a brief summary and relevant citiations links.",
    options=ClaudeAgentOptions(model=MODEL, allowed_tools=["WebSearch"]),
):
    print_activity(msg)
    messages.append(msg)

🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [4]:
display_agent_response(messages)

## 这里发生了什么：

- `query()` 创建一次单轮的智能体交互（没有对话记忆）
- `allowed_tools=["WebSearch"]` 授予 Claude 无需请求批准即可搜索网络的权限
- 智能体自主决定何时搜索、运行哪些查询，以及如何综合结果

**来自 `utils.agent_visualizer` 的可视化工具：**
- `print_activity()` - 实时显示智能体的动作（工具使用、思考）
- `display_agent_response()` - 把最终响应渲染为带样式的 HTML 卡片
- `visualize_conversation()` - 创建完整对话的时间线视图

就是这样！仅用几行代码就得到了一个可用的研究智能体。它会搜索相关信息、追踪有希望的线索，并给出带引用的综合摘要。

query（）函数创建一次无状态的智能体交互。每次调用都是独立的——没有对话记忆，也没有来自先前查询的上下文。这使它非常适合一次性研究任务：你需要快速得到答案，而不必维护状态。

**工具权限如何工作：**

`allowed_tools=["WebSearch"]` 参数授予 Claude 无需请求批准即可搜索的权限。这对自主运行至关重要：

- `Allowed tools` - Claude 可自由使用这些工具（本例中是 WebSearch）
- `Other tools` - 可用，但使用前需要批准
- `Read-only tools` - 像 Read 这样的工具默认始终允许
- `Disallowed tools` - 把工具加入 disallowed_tools，可将其完全从 Claude 的上下文中移除

**何时使用无状态查询：**

- 上下文无关紧要的一次性研究问题
- 并行处理相互独立的研究任务
- 希望每次查询都获得全新上下文的场景

**何时不应使用无状态查询：**

- 建立在先前发现之上的多轮调查
- 基于初步结果对研究进行迭代式细化
- 需要持续上下文的复杂分析

我们用 visualize_conversation 辅助函数来查看智能体实际做了什么：

In [5]:
from utils.agent_visualizer import visualize_conversation

visualize_conversation(messages)

## 从原型到生产环境：三项关键改进

我们的单行研究智能体能工作，但能力有限。没有记忆的单次查询无法处理迭代式研究（“先找到 X，再根据你发现的内容分析 Y”）。下面探讨三种可以进一步改进实现的方式。

**1. 用 ClaudeSDKClient 实现对话记忆**：无状态查询无法建立在先前发现之上。如果你先问“顶尖的 AI 初创公司有哪些？”，再问“它们如何获得融资？”，第二个查询完全不知道你指的是哪些公司。我们可以用 `ClaudeSDKClient` 在多次查询之间保持对话历史。


**2. 用系统提示词实现专业化行为**：研究领域通常有特定要求。财务分析对严谨性的要求与科技新闻摘要不同。使用系统提示词来固化你的研究标准、首选信息来源、引用格式或输出结构。参见我们的[智能体提示词指南](https://github.com/anthropics/anthropic-cookbook/tree/main/patterns/agents/prompts)中的研究专用示例。

**3. 用 Read 工具实现多模态研究**：真实的研究不只是文本。市场报告有图表，技术文档有示意图，竞争分析需要对比截图。启用 `Read` 工具，让 Claude 能够分析图片、PDF 和其他视觉内容。

我们来为研究智能体实现这三项改动。

In [6]:
from claude_agent_sdk import ClaudeSDKClient

# System prompt with citation requirements for research quality
RESEARCH_SYSTEM_PROMPT = """You are a research agent specialized in AI.

When providing research findings:
- Always include source URLs as citations
- Format citations as markdown links: [Source Title](URL)
- Group sources in a "Sources:" section at the end of your response"""

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="research_agent",
        system_prompt=RESEARCH_SYSTEM_PROMPT,
        allowed_tools=["WebSearch", "Read"],
        max_buffer_size=10 * 1024 * 1024,  # Increase to 10MB for image handling
    )
) as research_agent:
    # First query: Analyze the chart image
    await research_agent.query("Analyze the chart in research_agent/projects_claude.png")
    async for msg in research_agent.receive_response():
        print_activity(msg)
        messages.append(msg)

    # Second query: Use web search to validate/contextualize the chart findings
    await research_agent.query(
        "Based on the chart analysis, search for recent news or data that validates or provides context for these findings. Include source URLs."
    )
    async for msg in research_agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: Glob()
✓ Tool completed
🤖 Thinking...
🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


### 🔧 处理大型响应与缓冲区上限

在处理图片或大型数据时，你可能遇到缓冲区溢出错误：

```
Fatal error in message reader: Failed to decode JSON: JSON message exceeded maximum buffer size of 1048576 bytes
```

**产生原因：**
- 默认 `max_buffer_size` 为 1MB（1,048,576 字节）
- 图片在消息中以 base64 编码，体积显著增大
- 图表图片（磁盘上约 200KB）经 base64 编码后变为约 270KB 以上，再加上消息开销

**解决方案：** 在处理图片或大型工具输出时，把 `ClaudeAgentOptions` 中的 `max_buffer_size` 设为更高值（例如 10MB）。

**最佳实践：**
- 根据用例设置缓冲区大小：典型多模态工作用 10MB，处理大型文档时用更大的值
- 考虑是否真的需要传入完整图片——有时用文字描述或更小的缩略图就足够了
- 监控缓冲区错误并相应调整
- 在系统提示词中包含引用要求，以确保研究输出可核验

## 这里发生了什么：

这个示例结合了全部三项改进：对话记忆、带引用要求（citation-aware）的系统提示词，以及多模态分析。

**关键组成部分：**

| 组成部分 | 作用 |
|-----------|---------|
| `ClaudeSDKClient` | 在多次查询之间保持对话状态 |
| `RESEARCH_SYSTEM_PROMPT` | 强制引用格式和来源 URL |
| `allowed_tools=["WebSearch", "Read"]` | 启用网络搜索以及图片/文档分析 |
| `max_buffer_size=10MB` | 处理 base64 编码的图片而不溢出 |

**执行流程：**

1. **第一次查询** - 使用 `Read` 工具分析图表图片
2. **第一个响应循环** - 收集所有消息，直到智能体完成
3. **第二次查询** - 搜索网络以验证或补充图表所反映的情况
4. **上下文继承** - 第二次查询记得第一次对图表的分析

**为什么用 `ClaudeSDKClient` 而不是 `query()`：**

`async with ClaudeSDKClient()` 上下文管理器保持对话状态。每次 `receive_response()` 调用都建立在先前上下文之上。这与 `query()` 不同，后者创建相互独立、无状态的会话。

In [7]:
visualize_conversation(messages)

Pattern,Implication
Claude Code dominates Startup Work,Developers building products at startups prefer the code-focused interface for rapid development
Claude.ai leads in educational contexts,"The conversational nature of Claude.ai makes it more approachable for learning, research, and coursework"
Personal Projects are universal,Both platforms serve individual developers working on side projects equally well
Enterprise usage is balanced,Both products have found their place in professional enterprise environments
Metric,2025 Data
AI-generated/assisted code,41% of all code globally
Developers using AI coding assistants,82% daily or weekly
Market size projection,$30.1 billion by 2032
Google's AI-assisted code,25%
Chart Finding,Validation Status


## 面向生产环境构建

Jupyter 笔记本适合学习，但生产系统需要可复用的模块。我们已经把研究智能体封装进 `research_agent/agent.py`，并提供清晰的接口：

### 核心函数：

- `print_activity()` - 实时显示智能体正在做什么（从共享工具库导入）
- `get_activity_text()` - 提取活动文本供自定义处理程序使用，例如日志或监控
- `send_query()` - 研究查询的主入口，内置活动显示

### 内置的最佳实践：

该模块包含 `RESEARCH_SYSTEM_PROMPT`，确保：
- 来源 URL 始终作为引用包含在内
- 引用以 Markdown 链接格式呈现，便于干净渲染
- “Sources:” 小节把所有引用归为一组

### 显示控制：

`send_query()` 函数有一个 `display_result` 参数（默认：`True`）：
- `display_result=True` - 在 Jupyter 笔记本中渲染带样式的 HTML 卡片
- `display_result=False` - 仅返回文本结果，供程序化调用

这个智能体现在可以在任意 Python 脚本中使用！

适用于对话上下文无关紧要的独立问题。

该模块自动处理：
- 执行过程中的活动显示
- 为新对话重置上下文
- 最终响应的带样式 HTML 渲染

In [8]:
from research_agent.agent import send_query

# The module handles activity display, context reset, and result visualization internally
result = await send_query("What is the Claude Code SDK? Only do one websearch and be concise")

🤖 Using: WebSearch()
✓ Tool completed
🤖 Thinking...


现在我们测试一次复用同一会话的多轮对话。

多轮对话可以顺畅工作——只需传入 `continue_conversation=True`：

In [9]:
result1 = await send_query("What is Anthropic? Only do one websearch and be concise")

🤖 Using: WebSearch()
✓ Tool completed
🤖 Thinking...


In [10]:
# Continue the conversation to dig deeper by setting continue_conversation=True
result2 = await send_query(
    "What are some of their products?",
    continue_conversation=True,
)

🤖 Thinking...


## 结论

### 你构建了什么

在本 cookbook 中，你构建了三个逐步进阶的研究智能体：

- 无状态研究智能体 - 用于独立研究任务的单行查询
- 带记忆的有状态智能体 - 建立在先前发现之上的多轮调查
- 生产模块 - 可复用的研究函数，便于集成进应用程序

### 关键要点

**何时使用无状态查询（query（））：**

- 独立的研究问题
- 并行处理互不相关的任务
- 每次都需要全新上下文的场景

**何时使用有状态智能体（ClaudeSDKClient）：**

- 建立在先前发现之上的多轮调查
- 对研究进行迭代式细化
- 需要持续上下文的复杂分析

当信息不是自包含的、且最优工作流在探索过程中浮现时，研究智能体表现最好——竞争分析、技术故障排查、文献综述和调查性新闻都符合这一模式。

### 后续步骤

这套自主研究的基础为你准备企业级多智能体系统做好了铺垫。在下一个笔记本中，你将学习：

在协调智能体的指挥下调度专业化子智能体 通过护栏（hooks）和自定义命令实现治理 为不同受众（高管与技术团队）调整输出风格

下一步：[01_The_chief_of_staff_agent.ipynb](01_The_chief_of_staff_agent.ipynb) - 从单个智能体走向多智能体调度。